# Analyze 🤗 HuggingFace audio dataset

This notebook will show how to use DataChain for audio data, including how to:

- [Access audio data from the 🤗 HuggingFace storage.](#ingesting)
- [Apply emotion analysis model.](#emotions)
- [Apply Whisper speech recognition model to get transcripts.](#transcripts)
- [Join those data into a coherent dataset.](#joining)
- [Save the chain results as a dataset](#saving)
- [Filter and inspect the data.](#filter)

<a id='setup'></a>

## Setup

To start, install the dependencies and import the packages needed.

In [1]:
%pip install -q  "datachain[torch]" transformers hf_transfer


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch

from transformers import pipeline, Pipeline

import datachain as dc
from datachain import C, File, DataModel
from datachain.func import path

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'


/Users/ivan/Projects/datachain-examples/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<a id='ingesting'></a>

## Get data and create DataChain

We will use a dataset from https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0. You might need to register on Hugging Face, review and accept terms to access the dataset. We are using Hugging Face to show a possiblity of access Hugging Face file system directly, DataChain can work data in any major cloud - AWS and S3 compatible, Google Cloud, Azure.

To start, we will use data in `hf://datasets/mozilla-foundation/common_voice_17_0`. Specifically a subset from the `en/train/*.tar`: a folder of `tar` files, each representing a bulk of english voice recordings in `mp3`.

We can create a DataChain from the data source. DataChain can create a single dataset from many files in a storage bucket, folder, or other path. DataChain will generate one record per file, with each record including a `file` signal that saves metadata needed to read the file. In that way, DataChain keeps a link to the original storage without having to copy the files or load all file contents in memory.

### Indexing storage

To create a chain from a directory of files, use `dc.read_storage()` and point to the location of the directory. **Note:** it might take a while when you run it for the fist time since it's a pretty large Hugging Face repo.

In [3]:
dc.read_storage("hf://datasets/mozilla-foundation/common_voice_17_0").show(3)

,file,file
,path,size
0,mozilla-foundation/common_voice_17_0/.gitattri...,8318
1,mozilla-foundation/common_voice_17_0/README.md,12719
2,mozilla-foundation/common_voice_17_0/common_vo...,8194



[Limited by 3 rows]


Device set to use mps
Device set to use mps


DataChain created a record for each file in the repository, generating a `file` signal for each file. The file signal contains subsignals with metadata about each file, like `file.name` and `file.size`. You can define your own aggregate signals like this using Pydantic models, although this is outside the scope of this tutorial. You can also use the `file` signal to open and read the file as needed. We will come back to this later in the notebook.

### Basic filtering and TAR processing

You can see that as a result we are getting references to thefiles like `README` and also `.tar` archives that we can't use directly. We can use `filter`, `glob`, `gen` operations to get a chain with enginlish language individual entries. We'll apply filter by `file.path` and we'll use `process_tar` helper to create references to individual files inside those `tar` archives:

In [5]:
from datachain.lib.tar import process_tar

repo_dc = (
    dc.read_storage("hf://datasets/mozilla-foundation/common_voice_17_0")
        .settings(cache=True)
        .filter(C("file.path").glob("*/en/train/*.tar"))
  	    .limit(1)
        .gen(file=process_tar)
        .mutate(name=path.name(C("file.path")))
        .persist()
)

Let's preview the data. We see that we now have `mp3` file entries. Each entry is a virtual reference inside the original tar, no copying is happening.

In [6]:
repo_dc.select("name").show(3)

,name
0,common_voice_en_17924809.mp3
1,common_voice_en_19612700.mp3
2,common_voice_en_19612701.mp3



[Limited by 3 rows]


<a id='emotions'></a>

## Emotions classification

Now we can apply a model from Hugging Face to classify emotions in each individual `mp3` voice recording. We are running DataChain `map` operation which applies a custom defined function `def emotions()` and returns a `pydantic` model `Emotions` with the results. Each result is automaticaly saved as a set of additional columns in the DataChain table per each recording.

In [7]:
class Emotions(DataModel):
  anxiety: float = 0.0
  boredom: float = 0.0
  neutral: float = 0.0
  panic: float = 0.0
  sadness: float = 0.0
  shame: float = 0.0
  cold_anger: float = 0.0
  contempt: float = 0.0
  despair: float = 0.0
  disgust: float = 0.0
  elation: float = 0.0
  happy: float = 0.0
  hot_anger: float = 0.0
  interest: float = 0.0


def emotions(helper: Pipeline, file: File) -> Emotions:
  # `helper` is the Hugging Face pipeline object
  result = helper(file.read())
  return Emotions(
    **{emotion["label"].replace("-", "_"): emotion["score"] for emotion in result}
  )


emotions_dc = (
    repo_dc
       # Put `.settings(parallel=4)` on a bigger machine
  	  .limit(10)
      .setup(helper=lambda: pipeline(model="Krithika-p/my_awesome_emotions_model", device=device))
  	  .map(emotions=emotions)
      .persist()
)


Let's preview the emotions classifier results:

In [ ]:
emotions_dc.select("name", "emotions").show(3)

,name,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions
,,anxiety,boredom,neutral,panic,sadness,shame,cold_anger,contempt,despair,disgust,elation,happy,hot_anger,interest
0,common_voice_en_17924809.mp3,0.0,0.04572,0.0,0.0,0.0,0.0,0.138111,0.465626,0.0,0.093015,0.000000,0.000000,0.0,0.0
1,common_voice_en_19612700.mp3,0.0,0.00000,0.0,0.0,0.0,0.0,0.102818,0.132818,0.0,0.525119,0.029469,0.000000,0.0,0.0
2,common_voice_en_19612701.mp3,0.0,0.00000,0.0,0.0,0.0,0.0,0.097485,0.196979,0.0,0.435279,0.000000,0.027388,0.0,0.0



[Limited by 3 rows]


<a id='transcripts'></a>

## Getting transcriptions

Similar to the emotion classification above, we run another model to get transcriptions:

In [9]:
def text(helper: Pipeline, file: File) -> str:
  result = helper(file.read())
  return result["text"]


text_dc = (
    repo_dc
      # Put `.settings(parallel=4)` on a bigger machine
      .limit(10)
      # Make it large-v3 for better accuracy if you have a GPU / bigger machine 
      .setup(helper=lambda: pipeline("automatic-speech-recognition", "openai/whisper-tiny", torch_dtype=torch.float32, device=device))
  	  .map(text=text)
      .persist()
)


/Users/ivan/Projects/datachain-examples/.venv/lib/python3.12/site-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
Processed: 0 rows [00:00, ? rows/s]/Users/ivan/Projects/datachain-examples/.venv/lib/python3.12/site-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Processed: 2 rows [00:00, 10.42 rows/s]/Users/ivan/Projects/datachain-examples/.venv/lib/python3.12/s

Let's preview the transcripts results:

In [10]:
text_dc.select("name", "text").show(3)

,name,text
0,common_voice_en_17924809.mp3,Every evening the dogs in our neighborhood ar...
1,common_voice_en_19612700.mp3,I don't know that since been found.
2,common_voice_en_19612701.mp3,New York at the time had begun at the New Yor...



[Limited by 3 rows]


<a id='joining'></a>

## Merging the results into a single dataset

Two datachains (emotions and transcripts) could be merged into a single coherent dataset:


In [13]:
merged_dc = text_dc.merge(emotions_dc, on=C("file.path"), inner=True)
merged_dc.select("name", "text", "emotions").show(3)

,name,text,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions
,,,anxiety,boredom,neutral,panic,sadness,shame,cold_anger,contempt,despair,disgust,elation,happy,hot_anger,interest
0,common_voice_en_17924809.mp3,Every evening the dogs in our neighborhood ar...,0.016320,0.045720,0.012427,0.011692,0.015990,0.020466,0.138111,0.465626,0.008866,0.093015,0.010656,0.032714,0.010855,0.022805
1,common_voice_en_19612700.mp3,I don't know that since been found.,0.022328,0.010807,0.007961,0.020648,0.010606,0.008731,0.102818,0.132818,0.009250,0.525119,0.029469,0.023445,0.018201,0.010581
2,common_voice_en_19612701.mp3,New York at the time had begun at the New Yor...,0.025194,0.014919,0.007893,0.015892,0.011812,0.010603,0.097485,0.196979,0.010453,0.435279,0.021402,0.027388,0.014360,0.012324



[Limited by 3 rows]


<a id='saving'></a>

## Saving the chain

The result is saved using `DataChain.save()`. If name is provided this will save a persistent, versioned dataset that we can recover anytime in the future. It also will prevent DataChain from recomputing the same steps when running the chain again. If name is not provided it has an effect only on the current session to avoid recomputing certain data chains again and again, but it's not persisted.


In [ ]:
import datachain as dc
merged_dc.save("common-voice")
dc.read_dataset("common-voice").show(3)

AttributeError: module 'datachain' has no attribute 'save'

<a id='filter'></a>

## Filtering and inspecting results

Let's take a look at a few sample files from the dataset. `DataChain.filter()` allows for filtering of the dataset. In this case, we will filter to a single "happy" call using `emotions.happy` column:

In [ ]:
dc_chain = dc.read_dataset("common-voice")
sample = dc_chain.filter(C("emotions.happy") > 0.1).limit(1)

We can use `DataChain.collect()` to extract the values from the sample. Here's an example of collecting a subset of column signals from the sample:

In [ ]:
sample_results = list(sample.collect("file", "name", "text"))
sample_results

[(File(source='hf://datasets', path='mozilla-foundation/common_voice_17_0/audio/en/train/en_train_0.tar/en_train_0/common_voice_en_212609.mp3', size=47265, version='f843d7baced6e19aed4987754b00ce7689a85cac', etag='c49baf3b2be651a01f62d6e2d601429c', is_latest=True, last_modified=datetime.datetime(1970, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), location=[{'vtype': 'tar', 'parent': {'source': 'hf://datasets', 'path': 'mozilla-foundation/common_voice_17_0/audio/en/train/en_train_0.tar', 'size': 1799045120, 'version': 'f843d7baced6e19aed4987754b00ce7689a85cac', 'etag': 'a712e877031e919bb1d393d01c49895cf9b57bdf', 'is_latest': True, 'last_modified': '2024-04-04 18:32:42+00:00', 'location': None}, 'size': 47265, 'offset': 195072}]),
  'common_voice_en_212609.mp3',
  ' When can I see him?')]

The example has an output for each signal:
- `file` returns a special `File` object.
- `name` returns a short file name.
- `text` returns the trascript.

Let's now render and listen to this example:

In [ ]:
import IPython

file = next(sample.collect("file"))
IPython.display.Audio(file.read())